# Physical AI and Psychiatry: Synthetic Sandbox Demo

This notebook inspects the generated individual- and service-level outputs. All data are synthetic and assumption-driven; none of the results are clinical estimates.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('.')
OUT = ROOT / 'outputs'
FIG = ROOT / 'figures'

## Individual-level experiment

In [ ]:
ind = pd.read_csv(OUT / 'individual_arm_summary.csv')
ind.round(2)

In [ ]:
weekly = pd.read_csv(OUT / 'individual_weekly_trajectories.csv')
for arm, g in weekly.groupby('arm', sort=False):
    plt.plot(g['week'], g['mean_phq9_locf'], marker='o', label=arm)
plt.xlabel('Week'); plt.ylabel('Mean PHQ-9 (LOCF)'); plt.legend(frameon=False, fontsize=8); plt.show()

The robot arms include refusal, downtime, false alerts and physical near-misses. The hybrid arm adds human verification and therefore reduces confirmed false escalation at the cost of review time.

## Service-level experiment

In [ ]:
sys = pd.read_csv(OUT / 'system_scenario_summary.csv')
cols = ['scenario','mean_wait_minutes_mean','p90_wait_minutes_mean','throughput_per_day_mean','abandonment_percent_mean','false_alerts_per_100_arrivals_mean','missed_high_acuity_percent_mean']
sys[cols].round(2)

In [ ]:
rep = pd.read_csv(OUT / 'system_replication_results.csv')
order = rep['scenario'].drop_duplicates().tolist()
data = [rep.loc[rep.scenario == s, 'mean_wait_minutes'] for s in order]
plt.boxplot(data, tick_labels=['Manual','Screen','Robot','Robot + human gate'], showfliers=False)
plt.ylabel('Mean wait per replication (working minutes)'); plt.show()

## Promotion-gate questions

- Does the robot outperform a functionally matched screen agent, rather than only usual care?
- Is the incremental benefit robust to downtime, rework, sanitation, charging, and clinician review?
- Are consent, bystander privacy, therapeutic alliance, dependency, subgroup performance, cyber-physical safety, and decommissioning within prespecified bounds?
- Do all high-risk decisions remain traceable to an accountable human?

## Bootstrap confidence intervals and paired tests

Primary intervals are 95% BCa confidence intervals for the expected paired simulation contrast under fixed assumptions. Empirical replication ranges are reported separately and neither quantity is a clinical confidence interval.


In [ ]:
contrasts = pd.read_csv(OUT / 'individual_contrast_summary.csv')
contrasts[['comparator','metric_label','mean_difference','bootstrap_ci_low','bootstrap_ci_high','holm_sign_flip_pvalue','cohen_dz','mcse_mean_difference']].round(4)


## Sensitivity thresholds

In [ ]:
ind_grid = pd.read_csv(OUT / 'individual_sensitivity_grid.csv')
sys_grid = pd.read_csv(OUT / 'system_sensitivity_grid.csv')
pd.DataFrame({
    'criterion': ['Robot response > screen', 'Robot completion > screen', 'Robot wait <= screen'],
    'cells_met': [
        (ind_grid.response_difference_percentage_points > 0).sum(),
        (ind_grid.completion_difference_percentage_points > 0).sum(),
        (sys_grid.wait_difference_vs_screen_minutes <= 0).sum(),
    ],
    'cells_total': [len(ind_grid), len(ind_grid), len(sys_grid)]
})


## Verification

Run `python validate_models.py` from the project directory to check deterministic reproducibility, accounting identities, bounds, and impossible events.